In [ ]:
from matplotlib.lines import Line2D

role_df = df_stories.loc[
    mask_hammarby,
    ['player_id', 'player_name', 'role']
].copy()

role_df['player_name'] = role_df['player_name'].apply(lambda x: str(x).split()[-1])

player_name_to_role = (
    role_df[['player_name', 'role']]
    .dropna()
    .drop_duplicates('player_name')
    .set_index('player_name')['role']
    .to_dict()
)

scatter_df = scatter_df.copy()
scatter_df['role'] = scatter_df['player_name'].map(player_name_to_role)

def role_group(role):
    if pd.isna(role):
        return 'Unknown'

    r = str(role).lower()

    if 'keeper' in r or 'goal' in r:
        return 'Goalkeeper'
    elif 'back' in r or 'def' in r or 'cb' in r or 'fb' in r:
        return 'Defender'
    elif 'mid' in r or 'wing' in r:
        return 'Midfielder'
    elif 'forward' in r or 'striker' in r or 'att' in r or '9' in r:
        return 'Attacker'
    else:
        return 'Other'

scatter_df['role_group'] = scatter_df['role'].apply(role_group)

role_colors = {
    'Goalkeeper': '#9467bd',
    'Defender':   '#1f77b4',
    'Midfielder': '#2ca02c',
    'Attacker':   '#d62728',
    'Other':      '#ff7f0e',
    'Unknown':    '#7f7f7f'
}

fig, ax = pitch.grid(grid_height=0.9, title_height=0.06, axis=False,
                     endnote_height=0.04, title_space=0, endnote_space=0)
axp = ax['pitch']

pos = scatter_df.set_index('player_name')[['start_x', 'start_y']].to_dict('index')

max_pass = lines_df['pass_count'].max() if len(lines_df) else 1

tmp = lines_df.copy()
tmp['p1'] = tmp['pair_key'].str.split('_').str[0]
tmp['p2'] = tmp['pair_key'].str.split('_').str[1]

for _, row in tmp.iterrows():
    p1, p2 = row['p1'], row['p2']
    if (p1 not in pos) or (p2 not in pos):
        continue

    x1, y1 = pos[p1]['start_x'], pos[p1]['start_y']
    x2, y2 = pos[p2]['start_x'], pos[p2]['start_y']

    lw = (row['pass_count'] / max_pass) * 10
    pitch.lines(x1, y1, x2, y2, alpha=0.35, lw=lw, zorder=2,
                color='#ffcd00', ax=axp)

for role, group in scatter_df.groupby('role_group'):
    pitch.scatter(group['start_x'], group['start_y'],
                  s=group['marker_size'],
                  color=role_colors.get(role, '#7f7f7f'),
                  edgecolors='grey',
                  linewidth=1,
                  alpha=1,
                  ax=axp,
                  zorder=3)

xy = scatter_df[['start_x', 'start_y']].to_numpy(float)
names = scatter_df['player_name'].astype(str).to_list()
n = len(names)

diff = xy[:, None, :] - xy[None, :, :]
dist = np.sqrt((diff**2).sum(axis=2))
dist = dist + np.eye(n) * 1e9

nn = dist.argmin(axis=1)
vec = xy - xy[nn]
norm = np.linalg.norm(vec, axis=1, keepdims=True)
unit = np.divide(vec, norm, out=np.zeros_like(vec), where=norm > 0)

dmin = dist.min(axis=1)
offset_mag = np.where(dmin < 4, 5.0, np.where(dmin < 8, 3.5, 2.5))
offset = unit * offset_mag[:, None]

for i, name in enumerate(names):
    x, y = xy[i]
    dx, dy = offset[i]
    tx, ty = x + dx, y + dy

    tx = np.clip(tx, 0, 120)
    ty = np.clip(ty, 0, 80)

    ha = 'left' if dx >= 0 else 'right'

    pitch.annotate(name, xy=(tx, ty),
                   c='black', va='center', ha=ha, weight='bold', size=16,
                   ax=axp, zorder=6,
                   path_effects=[pe.withStroke(linewidth=4, foreground='white')])

# --- legend ---
legend_handles = [
    Line2D([0], [0], marker='o', color='w', label=role,
           markerfacecolor=color, markeredgecolor='grey', markersize=10)
    for role, color in role_colors.items()
    if role in scatter_df['role_group'].unique()
]

axp.legend(handles=legend_handles, loc='upper left', frameon=True)

plt.show()